# YOLOv8s AFPN + ATFL + Inner-IoU + DLQ Weighted on Kaggle

这份 notebook 用于在 Kaggle 上训练我们当前核心单模型之一：

- `YOLOv8s + AFPN`
- `ATFL`
- `Inner-IoU`
- `DLQ-Head (weighted)`

这版对应你之前的 `afpn_atfl_inneriou_dlq_weighted` 实验设置，重点是：

- 3 类任务：`chuck / gripper / drill_pipe`
- `DLQ-Head` 只作用在 `P3/P4`
- 推理分数使用 `cls_score * q`
- 开启 `drill_pipe-aware quality weighting`
- 方便直接在 Kaggle 上做短训或长训


In [ ]:
!nvidia-smi


In [ ]:
# 自动查找你上传到 Kaggle 的 3 类数据集根目录
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
candidate_roots = []
preferred_names = {
    'yolo-datasets-10k-yolov8-3class',
    'yolo-datasets-10k-yolov8-3cls',
    'yolo-datasets-10k-yolov8-3class-private',
}

for p in INPUT_ROOT.iterdir():
    if not p.is_dir():
        continue
    for q in [p] + [x for x in p.iterdir() if x.is_dir()]:
        if (q / 'images' / 'train').exists() and (q / 'labels' / 'train').exists():
            candidate_roots.append(q)

if not candidate_roots:
    raise FileNotFoundError('No YOLO 3-class dataset root found under /kaggle/input')

def score(path: Path):
    name = path.name.lower()
    preferred = 1 if name in preferred_names else 0
    return (preferred, len(name))

DATASET_ROOT = sorted(candidate_roots, key=score, reverse=True)[0]
print('DATASET_ROOT =', DATASET_ROOT)


In [ ]:
# 构建一个运行时数据集目录：图片软链接，标签复制到 /kaggle/working，顺手清理 cache
from pathlib import Path
import shutil
import os

RUNTIME_ROOT = Path('/kaggle/working/yolo_datasets_10k_yolov8_3class_runtime')
IMAGES_SRC = DATASET_ROOT / 'images'
LABELS_SRC = DATASET_ROOT / 'labels'

if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)

(RUNTIME_ROOT / 'images').mkdir(parents=True, exist_ok=True)
(RUNTIME_ROOT / 'labels').mkdir(parents=True, exist_ok=True)

for split in ['train', 'val', 'test']:
    img_src = IMAGES_SRC / split
    lb_src = LABELS_SRC / split
    img_dst = RUNTIME_ROOT / 'images' / split
    lb_dst = RUNTIME_ROOT / 'labels' / split

    if img_dst.exists() or img_dst.is_symlink():
        img_dst.unlink()
    os.symlink(img_src, img_dst, target_is_directory=True)

    shutil.copytree(lb_src, lb_dst)
    for cache_file in lb_dst.glob('*.cache'):
        cache_file.unlink()

print('RUNTIME_ROOT =', RUNTIME_ROOT)


In [ ]:
# 生成 Kaggle 训练用 data.yaml
from pathlib import Path

yaml_text = f'''path: {RUNTIME_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: chuck
  1: gripper
  2: drill_pipe
'''

DATA_CFG = Path('/kaggle/working/coalmine3_kaggle.yaml')
DATA_CFG.write_text(yaml_text, encoding='utf-8')
print(DATA_CFG.read_text(encoding='utf-8'))


In [ ]:
# 从 GitHub 拉取自定义源码，并以 editable 方式安装
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/tuanziya666/yolov8s_kuangjing.git'
REPO_DIR = Path('/kaggle/working/yolov8s_kuangjing')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'ultralytics'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

import ultralytics
print('python =', sys.executable)
print('ultralytics =', ultralytics.__file__)


In [ ]:
# 可选：清理旧结果
!rm -rf /kaggle/working/runs
!rm -rf /kaggle/working/evals
!rm -f /kaggle/working/yolov8s_afpn_atfl_inneriou_dlq_weighted_3cls_kaggle_results.zip


In [ ]:
# 公共训练参数
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EPOCHS = 300
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
PROJECT = '/kaggle/working/runs'
NAME = 'yolov8s_afpn_atfl_inneriou_dlq_weighted_3cls_kaggle'
PATIENCE = 100

DLQ_LEVELS = 'p3p4'
DLQ_LAMBDA = 0.2
DLQ_SCORE_MODE = 'mul'
DLQ_ALPHA = 0.6
USE_DRILL_QUALITY_WEIGHT = True
DRILL_QUALITY_REFINE = True
DLQ_TARGET_CLASS_IDS = '2'
DRILL_QUALITY_BASE_WEIGHT = 1.2
DRILL_QUALITY_SMALL_H1 = 0.06
DRILL_QUALITY_SMALL_H2 = 0.09
DRILL_QUALITY_SMALL_W1 = 1.3
DRILL_QUALITY_SMALL_W2 = 1.15

print('DATA_CFG =', DATA_CFG)
print('EPOCHS =', EPOCHS)
print('BATCH =', BATCH)
print('DEVICE =', DEVICE)
print('NAME =', NAME)


In [ ]:
# 训练前清理显存
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('cuda ready =', torch.cuda.is_available())


In [ ]:
# 开始训练：AFPN + ATFL + Inner-IoU + DLQ weighted
%cd /kaggle/working/yolov8s_kuangjing

import subprocess
import sys

cmd = [
    sys.executable,
    '-u',
    'train_yolov8s_afpn_atfl_inneriou_dlq_head.py',
    '--data', str(DATA_CFG),
    '--model', 'ultralytics/cfg/models/v8/yolov8s_afpn.yaml',
    '--pretrained', 'yolov8s.pt',
    '--epochs', str(EPOCHS),
    '--batch', str(BATCH),
    '--imgsz', str(IMGSZ),
    '--workers', str(WORKERS),
    '--device', str(DEVICE),
    '--optimizer', 'SGD',
    '--seed', str(SEED),
    '--deterministic', 'True',
    '--cache', 'False',
    '--amp', 'True',
    '--patience', str(PATIENCE),
    '--project', PROJECT,
    '--name', NAME,
    '--inner-ratio', '0.8',
    '--dlq-levels', DLQ_LEVELS,
    '--dlq-lambda', str(DLQ_LAMBDA),
    '--dlq-score-mode', DLQ_SCORE_MODE,
    '--dlq-alpha', str(DLQ_ALPHA),
    '--use-drill-quality-weight', str(USE_DRILL_QUALITY_WEIGHT),
    '--drill-quality-refine', str(DRILL_QUALITY_REFINE),
    '--dlq-target-class-ids', DLQ_TARGET_CLASS_IDS,
    '--drill-quality-base-weight', str(DRILL_QUALITY_BASE_WEIGHT),
    '--drill-quality-small-h1', str(DRILL_QUALITY_SMALL_H1),
    '--drill-quality-small-h2', str(DRILL_QUALITY_SMALL_H2),
    '--drill-quality-small-w1', str(DRILL_QUALITY_SMALL_W1),
    '--drill-quality-small-w2', str(DRILL_QUALITY_SMALL_W2),
]

print('Running command:')
print(' '.join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)


In [ ]:
# 检查训练输出
from pathlib import Path
import pandas as pd

run_dir = Path(PROJECT) / NAME
print('run_dir =', run_dir)
print('results.csv exists =', (run_dir / 'results.csv').exists())
print('best.pt exists =', (run_dir / 'weights' / 'best.pt').exists())
print('last.pt exists =', (run_dir / 'weights' / 'last.pt').exists())

if (run_dir / 'results.csv').exists():
    df = pd.read_csv(run_dir / 'results.csv')
    display(df.tail())


In [ ]:
# 用 best.pt 分别在 val / test 上评估
%cd /kaggle/working/yolov8s_kuangjing

from ultralytics import YOLO
from pathlib import Path

run_dir = Path(PROJECT) / NAME
best_pt = run_dir / 'weights' / 'best.pt'
eval_dir = Path('/kaggle/working/evals')
eval_dir.mkdir(parents=True, exist_ok=True)

model = YOLO(str(best_pt))

val_metrics = model.val(
    data=str(DATA_CFG),
    split='val',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(eval_dir),
    name=f'{NAME}_val',
)

test_metrics = model.val(
    data=str(DATA_CFG),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(eval_dir),
    name=f'{NAME}_test',
)

print('val mAP50-95 =', val_metrics.results_dict.get('metrics/mAP50-95(B)'))
print('test mAP50-95 =', test_metrics.results_dict.get('metrics/mAP50-95(B)'))


In [ ]:
# 打包结果，便于下载
%cd /kaggle/working
!zip -r yolov8s_afpn_atfl_inneriou_dlq_weighted_3cls_kaggle_results.zip runs evals coalmine3_kaggle.yaml
!ls -lh /kaggle/working/yolov8s_afpn_atfl_inneriou_dlq_weighted_3cls_kaggle_results.zip
